# Creating a Semantic Relationships Graph from an IFC file

In [1]:
# This cell is not needed if you have pip installed topologicpy
import sys
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")

## 1. Import the needed TopologicPy Classes

In [2]:
from topologicpy.Topology import Topology
from topologicpy.Graph import Graph
from topologicpy.Dictionary import Dictionary
from topologicpy.Helper import Helper

c:\Users\etmaglari\IAAC\etmaglari_gML\.gmlenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Check the TopologicPy version

In [3]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.22) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

## 4. Set default mappings (do not change key names)

In [5]:
# Do not change the code below this line
rels_color_mapping = {"IfcRelConnectsPathElements": "#440154",
                      "IfcRelContainedInSpatialStructure": "#31688E",
                      "IfcRelFillsElement": "#35B779",
                      "IfcRelSpaceBoundary": "#FDE725",
                      "IfcRelVoidsElement": "#E64B5D"
                      }

 
ifc_color_mapping = {"ifcbeam":"#440154",
                     "ifccovering": "#482878",
                     "ifcdoor": "#3E4989",
                     "ifcfooting": "#31688E",
                     "ifcfurnishingelement": "#26828E",
                     "ifcmember":"#1F9E89",
                     "ifcopeningelement":"#35B779",
                     "ifcrailing":"#6DCD59",
                     "ifcslab":"#B4DE2C",
                     "ifcspace":"#FDE725",
                     "ifcstairflight":"#FCA636",
                     "ifcwall":"#F8765C",
                     "ifcwallstandardcase":"#E64B5D",
                     "ifcwindow":"#D41159"
                     }


## 5. Specify the path to the IFC file and what to include

In [6]:
# Choose your own IFC file if you wish
ifc_file_path = r"C:\Users\etmaglari\IAAC\etmaglari_gML\S0 Classes\House_R25_detached.ifc"

## 6. Import the IFC file as graph

In [7]:
graph = Graph.ByIFCPath(ifc_file_path,
                        transferDictionaries=True,
                        storeBREP = True,
                        mode="geometry",
                        includeTypes = ["ifcWall"]
                       )


Graph.ByIFCFile - Warning: The relationship #154918=IfcRelAssociatesClassification('2Zspo8edhQlmUdSFep$PCU',#18,'Exterior Walls','Exterior Walls:B2010',(#694,#791,#840,#876,#910,#975,#2933,#3003,#3073,#3122,#3209,#6071,#6121,#6173,#6245,#32899,#32969,#33059,#34015,#39635,#39730,#39802,#39832,#39906,#39938,#40401,#42265,#42275,#42346,#42433,#42443,#42622,#42696,#54107,#59573,#59605,#59677,#59707,#60055,#60125,#60395,#60465,#60990,#65493,#65563,#65633,#65703,#65773,#65843,#65913,#65983,#66122,#66192,#66262,#66332,#66402,#71723,#152130,#152856,#153046,#153116,#153306,#153376,#153446,#153536,#153606,#154057,#154145),#736) is not supported. Skipping.
Graph.ByIFCFile - Warning: The relationship #154919=IfcRelAssociatesClassification('2FhH$nI$5hHMu3r_2wVhLw',#18,'Curtain Walls','Curtain Walls:B2020200',(#6264,#6463,#33078,#33506,#34034,#39464,#59299,#59419,#60144,#60484,#60684,#61009,#152875,#153135,#153625,#153869),#6459) is not supported. Skipping.
Graph.ByIFCFile - Warning: The relationshi

## 7. Extract breps from the vertices and set colours for vertices and edges

In [8]:
vertices = Graph.Vertices(graph)
boxes = []
for v in vertices:
    d = Topology.Dictionary(v)
    brep = Dictionary.ValueAtKey(d, "BREP")
    topology = Topology.ByBREPString(brep)
    box = Topology.BoundingBox(topology)
    boxes.append(box)
    ifc_type = Dictionary.ValueAtKey(d, "IFC_type", "Unknown")
    vertexColor = ifc_color_mapping.get(ifc_type, "white")
    d = Dictionary.SetValuesAtKeys(d, ["size", "color"], [10, vertexColor])
    v = Topology.SetDictionary(v, d)

edges = Graph.Edges(graph)
rels = []
for e in edges:
    d = Topology.Dictionary(e)
    ifc_rel = Dictionary.ValueAtKey(d, "IFC_type")
    edgeColor = rels_color_mapping.get(ifc_rel, "white")
    d = Dictionary.SetValuesAtKeys(d, ["width", "color"], [3, edgeColor])
    e = Topology.SetDictionary(e, d)

## 8. Show the final result

In [9]:
Topology.Show(boxes, graph,
              faceOpacity=0.1,
              sagitta=0.15,
              absolute=False,
              backgroundColor="black",
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              width=800,
              height=600,
              renderer=renderer)

## 9. Use Pyvis for an alternative visualisation

In [ ]:
pyvis_graph = Graph.PyvisGraph(graph, path=r"C:\Users\sarwj\OneDrive - Cardiff University\Desktop\pyvis_graph.html",
                               vertexSizeKey="size",
                               vertexColorKey="color",
                               vertexLabelKey="IFC_type",
                               edgeWeightKey="width",
                               edgeColorKey="color")